# RECOVERY CELL
Run this after notebook interruption. Hardcodes completed results and runs only what is missing.

**IMPORTANT:** Make sure the baseline model is still loaded in memory.
If the runtime was reset, run Cell 1 (imports), Cell 2 (data loading), Cell 3 (model definition),
then load the saved baseline before running this.

In [ ]:
# ── STEP 0: Check if baseline model is still in memory ───────────────────────
# If you see an error here, run the import/data/model cells first,
# then run the cell below this one to reload the baseline from Drive.

try:
    acc_check = evaluate(baseline_model, testloader)
    print(f'Baseline model still in memory. Accuracy: {acc_check:.2f}%')
    BASELINE_IN_MEMORY = True
except NameError:
    print('Baseline model NOT in memory. Run the reload cell below first.')
    BASELINE_IN_MEMORY = False

In [ ]:
# ── STEP 0b: ONLY run this if baseline is NOT in memory ──────────────────────
# (i.e., the runtime was reset)

if not BASELINE_IN_MEMORY:
    from google.colab import drive
    drive.mount('/content/drive')

    import os, torch, numpy as np, random, copy, time, json
    import torchvision, torchvision.transforms as transforms
    import torch.nn as nn
    import torch.optim as optim
    from scipy.linalg import eigh
    from scipy.stats import pearsonr
    from collections import defaultdict
    import warnings
    warnings.filterwarnings('ignore')

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SAVE_DIR = '/content/drive/MyDrive/AEI_Experiments/CIFAR100'
    NUM_CLASSES = 100
    SEED = 42
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

    # ── Reload transforms and loaders ────────────────────────────────────────
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761)),
    ])
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761)),
    ])
    trainset = torchvision.datasets.CIFAR100(
        root='./data', train=True, download=True, transform=transform_train)
    testset  = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=transform_test)
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=128, shuffle=True, num_workers=2)
    testloader  = torch.utils.data.DataLoader(
        testset,  batch_size=256, shuffle=False, num_workers=2)
    subset_dataset = torch.utils.data.Subset(trainset, list(range(5000)))
    activationloader = torch.utils.data.DataLoader(
        subset_dataset, batch_size=256, shuffle=False, num_workers=2)

    # ── Model definition ─────────────────────────────────────────────────────
    class SimpleCNN(nn.Module):
        def __init__(self, num_classes=100, conv2_filters=64):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(32, conv2_filters, kernel_size=3, padding=1)
            self.pool  = nn.MaxPool2d(2, 2)
            self.relu  = nn.ReLU()
            self.fc_input_size = conv2_filters * 8 * 8
            self.fc1 = nn.Linear(self.fc_input_size, 256)
            self.fc2 = nn.Linear(256, num_classes)
        def forward(self, x):
            x = self.pool(self.relu(self.conv1(x)))
            x = self.pool(self.relu(self.conv2(x)))
            x = x.view(x.size(0), -1)
            x = self.relu(self.fc1(x))
            return self.fc2(x)

    def evaluate(model, loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                _, predicted = model(inputs).max(1)
                correct += predicted.eq(targets).sum().item()
                total   += inputs.size(0)
        return 100. * correct / total

    def train_epoch(model, loader, optimizer, criterion):
        model.train()
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()

    # ── Load baseline from Drive ──────────────────────────────────────────────
    baseline_model = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)
    baseline_model.load_state_dict(
        torch.load(os.path.join(SAVE_DIR, 'baseline_cifar100.pth'),
                   map_location=DEVICE))
    baseline_acc = evaluate(baseline_model, testloader)
    print(f'Baseline loaded. Accuracy: {baseline_acc:.2f}%')
    BASELINE_IN_MEMORY = True

In [ ]:
# ── STEP 1: Hardcode all COMPLETED results ────────────────────────────────────
from collections import defaultdict
import numpy as np

# These are exactly the numbers from your output
results = defaultdict(dict)

results['Spectral (AEI)'][0.20] = 49.95
results['Spectral (AEI)'][0.30] = 49.26
results['Spectral (AEI)'][0.40] = 48.60

results['Hybrid (AEI×L2)'][0.20] = 50.64
results['Hybrid (AEI×L2)'][0.30] = 50.06
results['Hybrid (AEI×L2)'][0.40] = 50.21

results['L1 Norm'][0.20] = 51.46
results['L1 Norm'][0.30] = 51.44
results['L1 Norm'][0.40] = 50.12

results['L2 Norm'][0.20] = 51.49
results['L2 Norm'][0.30] = 51.51
results['L2 Norm'][0.40] = 49.94

results['SNIP'][0.20] = 51.32
results['SNIP'][0.30] = 51.40
results['SNIP'][0.40] = 50.86

results['GraSP'][0.20] = 46.11
results['GraSP'][0.30] = 43.36
results['GraSP'][0.40] = 41.00

# Random 20% already done
results['Random'][0.20] = (48.59, 0.46)   # (mean, std)

baseline_acc = 51.43

print('Completed results loaded into memory.')
print('Still need: Random 30% and Random 40%')

In [ ]:
# ── STEP 2: Define pruning helpers (needed for Random trials) ─────────────────

def prune_and_finetune_random(model_original, sparsity,
                               num_classes=100, finetune_epochs=20):
    """
    Random pruning: randomly selects which filters to keep,
    then fine-tunes. Returns post-finetune accuracy.
    """
    F_orig = model_original.conv2.out_channels
    n_keep = max(1, int(F_orig * (1.0 - sparsity)))

    # Random selection of filters to keep
    keep_idx = np.sort(np.random.choice(F_orig, n_keep, replace=False))

    # Build pruned model
    pruned_model = SimpleCNN(num_classes=num_classes,
                             conv2_filters=n_keep).to(DEVICE)

    # Copy weights
    pruned_model.conv1.weight.data = model_original.conv1.weight.data.clone()
    pruned_model.conv1.bias.data   = model_original.conv1.bias.data.clone()
    pruned_model.conv2.weight.data = model_original.conv2.weight.data[keep_idx].clone()
    pruned_model.conv2.bias.data   = model_original.conv2.bias.data[keep_idx].clone()

    spatial = 8 * 8
    fc1_keep = np.concatenate(
        [np.arange(k * spatial, (k+1) * spatial) for k in keep_idx])
    pruned_model.fc1.weight.data = model_original.fc1.weight.data[:, fc1_keep].clone()
    pruned_model.fc1.bias.data   = model_original.fc1.bias.data.clone()
    pruned_model.fc2.weight.data = model_original.fc2.weight.data.clone()
    pruned_model.fc2.bias.data   = model_original.fc2.bias.data.clone()

    # Fine-tune
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pruned_model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=finetune_epochs)

    for epoch in range(finetune_epochs):
        train_epoch(pruned_model, trainloader, optimizer, criterion)
        scheduler.step()

    return evaluate(pruned_model, testloader)


print('Pruning helpers defined.')

In [ ]:
# ── STEP 3: Run ONLY the missing Random trials ────────────────────────────────
# Random 20% is already done (48.59% ± 0.46%)
# We only need 30% and 40%

N_RANDOM_TRIALS = 5
MISSING_SPARSITIES = [0.30, 0.40]

print('Running missing Random trials...')
print('(Each sparsity level = 5 trials × 20 fine-tune epochs)')
print()

for sparsity in MISSING_SPARSITIES:
    trial_accs = []
    t0 = time.time()

    for trial in range(N_RANDOM_TRIALS):
        acc = prune_and_finetune_random(
            baseline_model, sparsity, num_classes=NUM_CLASSES)
        trial_accs.append(acc)
        print(f'  Sparsity {int(sparsity*100)}% | Trial {trial+1}/{N_RANDOM_TRIALS}: {acc:.2f}%')

    mean_acc = np.mean(trial_accs)
    std_acc  = np.std(trial_accs)
    results['Random'][sparsity] = (mean_acc, std_acc)
    elapsed = time.time() - t0
    print(f'  → Sparsity {int(sparsity*100)}%: {mean_acc:.2f}% ± {std_acc:.2f}% '
          f'[{elapsed/60:.1f} min]\n')

print('Random pruning complete!')

In [ ]:
# ── STEP 4: Print the full final results table ────────────────────────────────

METHODS       = ['Spectral (AEI)', 'Hybrid (AEI×L2)',
                 'L1 Norm', 'L2 Norm', 'SNIP', 'GraSP', 'Random']
SPARSITY_LEVELS = [0.20, 0.30, 0.40]

print('='*65)
print('TABLE: Fine-tuned Accuracy on CIFAR-100')
print('(SimpleCNN, conv2 filter pruning)')
print('='*65)
print(f'{"Method":<22} {"20%":>10} {"30%":>10} {"40%":>10}')
print('-'*55)

for method in METHODS:
    row = f'{method:<22}'
    for sparsity in SPARSITY_LEVELS:
        val = results[method][sparsity]
        if isinstance(val, tuple):
            row += f'  {val[0]:.2f}±{val[1]:.2f}'
        else:
            row += f'  {val:.2f}%   '
    print(row)

print('-'*55)
print(f'{"Baseline":<22}  {baseline_acc:.2f}%  (all sparsities)')

# ── Key observations ─────────────────────────────────────────────────────────
print()
print('KEY OBSERVATIONS:')

hybrid_40  = results['Hybrid (AEI×L2)'][0.40]
l2_40      = results['L2 Norm'][0.40]
aei_40     = results['Spectral (AEI)'][0.40]
grasp_40   = results['GraSP'][0.40]
random_40  = results['Random'][0.40]

if isinstance(random_40, tuple):
    random_40_val = random_40[0]
else:
    random_40_val = random_40

print(f'  Hybrid vs L2 at 40%:  {hybrid_40:.2f}% vs {l2_40:.2f}% '
      f'(Hybrid {"BEATS" if hybrid_40 > l2_40 else "trails"} L2 by {abs(hybrid_40-l2_40):.2f}pp)')
print(f'  AEI gap vs L2 at 40%: {l2_40 - aei_40:.2f}pp')
print(f'  Hybrid gap vs L2 at 40%: {l2_40 - hybrid_40:.2f}pp')
print(f'  Gap narrowing (AEI→Hybrid): {(l2_40-aei_40):.2f}pp → {(l2_40-hybrid_40):.2f}pp')
print(f'  GraSP collapse at 40%: {baseline_acc - grasp_40:.2f}pp below baseline')
print(f'  AEI vs Random at 40%: {aei_40:.2f}% vs {random_40_val:.2f}%')

# ── FLOPs table ──────────────────────────────────────────────────────────────
print()
print('TABLE: FLOPs reduction (SimpleCNN, CIFAR-100)')
print(f'{"Sparsity":<12} {"Filters":>8} {"FLOPs":>10} {"Reduction":>12}')
print('-'*45)
print(f'{"Baseline":<12} {64:>8} {13.36:>9.2f}M {"--":>12}')
flops_data = [(0.20, 51, 11.01), (0.30, 44, 9.75), (0.40, 38, 8.67)]
for s, filters, flops in flops_data:
    reduction = 100 * (1 - flops / 13.36)
    print(f'{int(s*100):>3}%         {filters:>8} {flops:>9.2f}M {reduction:>10.1f}%')

In [ ]:
# ── STEP 5: Save results to Drive ────────────────────────────────────────────
import json, os

SAVE_DIR = '/content/drive/MyDrive/AEI_Experiments/CIFAR100'

results_json = {}
for method, sparsity_dict in results.items():
    results_json[method] = {}
    for sparsity, val in sparsity_dict.items():
        if isinstance(val, tuple):
            results_json[method][str(sparsity)] = {
                'mean': round(val[0], 4),
                'std':  round(val[1], 4)
            }
        else:
            results_json[method][str(sparsity)] = round(val, 4)

summary = {
    'dataset':        'CIFAR-100',
    'architecture':   'SimpleCNN (Conv1:3→32, Conv2:32→64, FC1:4096→256, FC2:256→100)',
    'pruning_target': 'Conv2 filters',
    'baseline_acc':   baseline_acc,
    'baseline_flops_M': 13.36,
    'results':        results_json,
}

path = os.path.join(SAVE_DIR, 'cifar100_results_summary.json')
with open(path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Results saved to {path}')
print('\nAll done! You can now run the visualization cells from the main notebook.')

In [ ]:
# ── STEP 6: Quick spectral histogram (needs AEI scores) ──────────────────────
# Only run this if R_scores is still in memory from the original session.
# If not, we can skip this — the table results are the important part.

try:
    _ = R_scores
    print('R_scores found in memory — spectral plot available.')
    print('Run the spectral histogram cell from the main notebook.')
except NameError:
    print('R_scores not in memory (runtime was reset).')
    print('To regenerate the spectral plot, recompute AEI scores:')
    print("  R_scores, A_graph, fiedler_vec, overhead = get_aei_scores(baseline_model, activationloader)")
    print('Then load keep_indices from saved pruned model files.')
    print()
    print('For the paper, the results TABLE is the critical output.')
    print('The spectral plot can be regenerated separately if needed.')